In [2]:
import findspark
findspark.init("/opt/spark")

In [5]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [4]:
spark = SparkSession.builder.appName("Simple Text Classification").master("local[*]").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/28 03:29:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
# Sample Dataset (replace with your actual data)
data = [("I loved this movie! Great acting and story.", "positive"),
        ("This was the worst movie I've ever seen.", "negative"),
        ("Decent movie, but could have been better.", "positive")]
df = spark.createDataFrame(data, ["review", "label_raw"])

In [7]:
df.limit(5).toPandas()

,review,label_raw
0,I loved this movie! Great acting and story.,positive
1,This was the worst movie I've ever seen.,negative
2,"Decent movie, but could have been better.",positive


In [8]:
tokenizer = Tokenizer(inputCol="review", outputCol="words")

In [9]:
df2 = tokenizer.transform(df)

In [10]:
df2.limit(5).toPandas()

,review,label_raw,words
0,I loved this movie! Great acting and story.,positive,"[i, loved, this, movie!, great, acting, and, s..."
1,This was the worst movie I've ever seen.,negative,"[this, was, the, worst, movie, i've, ever, seen.]"
2,"Decent movie, but could have been better.",positive,"[decent, movie,, but, could, have, been, better.]"


In [11]:
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures")

In [12]:
df3 = hashingTF.transform(df2)

In [13]:
df3.limit(5).toPandas()

,review,label_raw,words,rawFeatures
0,I loved this movie! Great acting and story.,positive,"[i, loved, this, movie!, great, acting, and, s...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,This was the worst movie I've ever seen.,negative,"[this, was, the, worst, movie, i've, ever, seen.]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,"Decent movie, but could have been better.",positive,"[decent, movie,, but, could, have, been, better.]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [14]:
df3.printSchema()

root
 |-- review: string (nullable = true)
 |-- label_raw: string (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rawFeatures: vector (nullable = true)



In [15]:
idf = IDF(inputCol="rawFeatures", outputCol="features")

In [16]:
df4 = idf.fit(df3).transform(df3)

In [17]:
df4.limit(5).toPandas()

24/07/28 03:32:47 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:32:47 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB


,review,label_raw,words,rawFeatures,features
0,I loved this movie! Great acting and story.,positive,"[i, loved, this, movie!, great, acting, and, s...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,This was the worst movie I've ever seen.,negative,"[this, was, the, worst, movie, i've, ever, seen.]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,"Decent movie, but could have been better.",positive,"[decent, movie,, but, could, have, been, better.]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [18]:
label_indexer = StringIndexer().setHandleInvalid("skip") \
.setInputCol("label_raw") \
.setOutputCol("label")

In [19]:
df5 = label_indexer.fit(df4).transform(df4)

In [20]:
df5.limit(5).toPandas()

24/07/28 03:33:50 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:33:51 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB


,review,label_raw,words,rawFeatures,features,label
0,I loved this movie! Great acting and story.,positive,"[i, loved, this, movie!, great, acting, and, s...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0
1,This was the worst movie I've ever seen.,negative,"[this, was, the, worst, movie, i've, ever, seen.]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0
2,"Decent movie, but could have been better.",positive,"[decent, movie,, but, could, have, been, better.]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0


In [21]:
# Logistic Regression Model
lr = LogisticRegression(maxIter=10, regParam=0.01)

In [22]:
df6 = lr.fit(df5).transform(df5)

24/07/28 03:34:51 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:34:53 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/07/28 03:34:55 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:35:02 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:35:05 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:35:08 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:35:11 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:35:14 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:35:17 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:35:19 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:35:22 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:35:24 WARN DAGSchedul

In [23]:
df6.limit(5).toPandas()

24/07/28 03:35:46 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 03:35:46 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB


,review,label_raw,words,rawFeatures,features,label,rawPrediction,probability,prediction
0,I loved this movie! Great acting and story.,positive,"[i, loved, this, movie!, great, acting, and, s...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,"[5.328140565174225, -5.328140565174225]","[0.9951703530660029, 0.004829646933997123]",0.0
1,This was the worst movie I've ever seen.,negative,"[this, was, the, worst, movie, i've, ever, seen.]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,"[-4.676493883486264, 4.676493883486264]","[0.009225698239070596, 0.9907743017609294]",1.0
2,"Decent movie, but could have been better.",positive,"[decent, movie,, but, could, have, been, better.]","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.0,"[5.441092011687838, -5.441092011687838]","[0.9956839618431035, 0.004316038156896518]",0.0
